In [ ]:
#NO se corrio 
from dotenv import load_dotenv

load_dotenv()

In [2]:
#modificada para local 
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model

In [ ]:
#use Linux NO se corrio
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [ ]:
#NO se corrio 
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [3]:
print(sys.executable)

<repo>/.venv/bin/python


In [4]:
#modificada para local 
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command": sys.executable,          # el python del .venv, no el del sistema
            "args": ["resources/2.1_mcp_server.py"],
        }
    }
)

In [5]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [ ]:
# NO se corrio
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=prompt
)

In [7]:
# Modificada para el local con tool caling
from langchain.agents import create_agent
agent = create_agent(
    model=get_model("gemma4:latest"),
    tools=tools,
    system_prompt=prompt
)

In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [9]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='eb83c66a-530f-41d6-99e1-d6a154fab463'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:latest', 'created_at': '2026-10-04T23:12:29.9124927Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11736408600, 'load_duration': 6727612900, 'prompt_eval_count': 208, 'prompt_eval_duration': 3906710000, 'eval_count': 22, 'eval_duration': 1097504000, 'logprobs': None, 'model_name': 'gemma4:latest', 'model_provider': 'ollama'}, id='lc_run--01a10930-88f5-70c2-ba11-8889688e13a8-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': '9503d8d6-fb10-4378-add8-9b8bc2a11937', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 208, 'output_tokens': 22, 'total_tokens': 230}),
              ToolMessage(content=[{'type': 'text', 'text': '{\n  "query": "la

## Online MCP

In [ ]:
# NO se corrio 
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [ ]:
client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command": "uv", # UV 
            "args": ["run", "python", "resources/2.1_mcp_server.py"],
        }
    }
)

In [10]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": sys.executable, # el python del .venv, no el del sistema
            "args": ["-m", "mcp_server_time", "--local-timezone=America/Bogota"],
        }
    }
)
tools = await client.get_tools()

In [ ]:
# NO se corrio 
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [11]:
agent = create_agent(
    model=get_model("gemma4:latest"),
    tools=tools,
    system_prompt=prompt
)

In [12]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='cb90cc68-7256-4852-8606-4f6517cba094'),
              AIMessage(content='What timezone would you like to know the current time for?', additional_kwargs={}, response_metadata={'model': 'gemma4:latest', 'created_at': '2026-10-04T23:17:10.1795827Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9284047400, 'load_duration': 8380700, 'prompt_eval_count': 435, 'prompt_eval_duration': 7917716000, 'eval_count': 13, 'eval_duration': 1321467000, 'logprobs': None, 'model_name': 'gemma4:latest', 'model_provider': 'ollama'}, id='lc_run--01a10934-d96e-78e3-a668-17aae97e60ab-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 435, 'output_tokens': 13, 'total_tokens': 448})]}
